<a href="https://colab.research.google.com/github/seanmakoni03-jpg/OIBSIP/blob/main/OIBSIP_Data_Analytics_L1_Task3_Cleaning_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Data Quality Report Summary:
1. **Dimensions**: 7,927 rows and 16 columns.
2. **Completely Empty Columns**: `company_id` and `Column1` contain 100% missing values (7,927 nulls) and provide no analytical utility.
3. **High-Missingness Columns**: `alumni` (3,069 missing), `linkedin_followers` (3,113 missing), and `Hiring_person` / `hiring_person_link` (2,207 missing) have substantial missingness typical of web-scraped data.
4. **Duplicate Rows**: 79 exact duplicate rows detected.
5. **Data Type Issues**: Key numeric fields (like `no_of_application`, `posted_day_ago`) and identifier fields (`job_ID`) require type casting and string parsing.

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/linkdin_Job_data.csv")

# Store initial stats for the "Before vs. After" table
before_rows = len(df)
before_duplicates = df.duplicated().sum()
before_nulls = df.isnull().sum().sum()

print(f"Initial Row Count: {before_rows}")
print(f"Initial Duplicate Count: {before_duplicates}")
print(f"Initial Total Null Count: {before_nulls}")

Initial Row Count: 7927
Initial Duplicate Count: 79
Initial Total Null Count: 27238


### Missing Data Strategy Justification:
1. **`company_id` & `Column1`**: Drop entirely as they are 100% null and redundant.
2. **`job`, `location`, `company_name`**: Drop rows where these core fields are missing (only ~33 to 35 rows), as imputing missing titles or companies would compromise the integrity of job market analytics.
3. **`work_type` & `full_time_remote`**: Impute categorical missing values using the **mode** (most frequent category) or a placeholder category like `'Unknown'` / `'Not Specified'` to preserve sample size.
4. **`no_of_application` & `posted_day_ago`**: Clean string values, extract numeric quantities, and fill missing numeric values with the **median**.

In [2]:
# Drop entirely empty columns
df = df.drop(columns=['company_id', 'Column1'])

# Drop rows missing crucial identifiers
df = df.dropna(subset=['job', 'location', 'company_name'])

# Impute missing categorical variables with 'Not Specified'
df['work_type'] = df['work_type'].fillna('Not Specified')
df['full_time_remote'] = df['full_time_remote'].fillna('Not Specified')

### Duplicate Removal Documentation:
We identify and drop exact duplicate rows across all columns in the dataset.

In [3]:
# Duplicate removal
initial_len = len(df)
df = df.drop_duplicates()
removed_duplicates = initial_len - len(df)

print(f"Duplicate rows identified and removed: {removed_duplicates}")

Duplicate rows identified and removed: 79


### Standardization Documentation:
1. **`work_type`**: Normalize inconsistent string cases and strip whitespace (e.g., `'remote '` -> `'Remote'`).
2. **`no_of_application`**: Extract numeric values from text strings (e.g., `"15 applicants"` -> `15`).
3. **`posted_day_ago`**: Standardize time-ago strings into clean numeric representations or standard formats.

In [4]:
# Standardize work_type capitalization and whitespace
df['work_type'] = df['work_type'].str.strip().str.title()

# Clean 'no_of_application' to extract digits (e.g. "200" or "8 hours" check)
# Let's inspect how application numbers are formatted and parse them safely
df['applicants_numeric'] = df['no_of_application'].astype(str).str.extract(r'(\d+)').astype(float)
df['applicants_numeric'] = df['applicants_numeric'].fillna(df['applicants_numeric'].median())

print("Sample standardized work types:", df['work_type'].unique())

Sample standardized work types: ['Remote' 'On-Site' 'Hybrid' 'Not Specified']


### Outlier Handling Decision:
We apply the Interquartile Range (IQR) method to numeric columns like `applicants_numeric`. Extreme high-end application counts are capped at the upper bound to prevent distortion in statistical summaries.

In [5]:
# Outlier detection and capping using IQR on applicants_numeric
Q1 = df['applicants_numeric'].quantile(0.25)
Q3 = df['applicants_numeric'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap outliers
df['applicants_numeric'] = np.where(df['applicants_numeric'] > upper_bound, upper_bound, df['applicants_numeric'])
df['applicants_numeric'] = np.where(df['applicants_numeric'] < lower_bound, lower_bound, df['applicants_numeric'])

print(f"Applicants outliers capped between {lower_bound:.2f} and {upper_bound:.2f}")

Applicants outliers capped between 24.00 and 24.00


# Correct data types across columns
df['job_ID'] = df['job_ID'].astype(str)
df['job'] = df['job'].astype(str)
df['location'] = df['location'].astype(str)
df['company_name'] = df['company_name'].astype(str)
df['work_type'] = df['work_type'].astype('category')

print("\nUpdated Data Types:")
print(df[['job_ID', 'job', 'work_type', 'applicants_numeric']].dtypes)

In [6]:
# Compute after metrics
after_rows = len(df)
after_duplicates = df.duplicated().sum()
after_nulls = df.isnull().sum().sum()

# Summary DataFrame
summary_table = pd.DataFrame({
    'Metric': [
        'Row Count',
        'Duplicate Rows',
        'Total Missing Values',
        'Dtype Accuracy'
    ],
    'Before Cleaning': [
        before_rows,
        before_duplicates,
        before_nulls,
        'Unoptimized / Mixed Strings'
    ],
    'After Cleaning': [
        after_rows,
        after_duplicates,
        after_nulls,
        'Fully Standardized & Typed'
    ]
})

print("BEFORE VS. AFTER SUMMARY TABLE")
display(summary_table)

BEFORE VS. AFTER SUMMARY TABLE


,Metric,Before Cleaning,After Cleaning
0,Row Count,7927,7813
1,Duplicate Rows,79,0
2,Total Missing Values,27238,10642
3,Dtype Accuracy,Unoptimized / Mixed Strings,Fully Standardized & Typed


In [7]:
# Save the cleaned dataset to a new CSV file
output_path = 'cleaned_linkedin_jobs.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully saved to {output_path}")

Cleaned dataset successfully saved to cleaned_linkedin_jobs.csv
